# FIN exact algebra — Singular campaign
This notebook uses a normal Python kernel for portable input construction and **Singular** for the actual P485 Gröbner reduction and P487 elimination. Default: `P486 → PREPARE → P485 → P487`. P475 is deliberately separate.

The exact normalization `sqrt(2)=2-4*alpha^2`, `alpha=sin(pi/8)`, is audited before Singular receives the ideal.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, zipfile, hashlib, json
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').exists()
print({'python':sys.version, 'colab':IN_COLAB, 'kaggle':IN_KAGGLE, 'Singular':shutil.which('Singular') or shutil.which('singular')})

## Install Singular if necessary
Colab/Kaggle can try the system package. In a SageMath environment Singular is normally already included. Installation requires network/package access and may not be available on every runtime image.

In [ ]:
INSTALL_SINGULAR_IF_MISSING = True
singular = shutil.which('Singular') or shutil.which('singular')
if singular is None and INSTALL_SINGULAR_IF_MISSING:
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','singular'], check=True)
    singular = shutil.which('Singular') or shutil.which('singular')
if singular is None:
    raise RuntimeError('Singular is unavailable. Use a SageMath runtime or install the Singular package.')
print(subprocess.check_output([singular,'--version'], text=True).splitlines()[0])

## Persistent work directory
For long Colab calculations set `USE_GOOGLE_DRIVE=True`. Singular itself is CPU/RAM intensive; a GPU does not accelerate this computation.

In [ ]:
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FIN_Singular_campaign')
elif IN_KAGGLE:
    ROOT = Path('/kaggle/working/FIN_Singular_campaign')
else:
    ROOT = Path.cwd()/'FIN_Singular_campaign'
ROOT.mkdir(parents=True, exist_ok=True)
print(ROOT)

## Upload/extract the verified bundle

In [ ]:
BUNDLE_NAME='FIN_Exact_Algebra_Singular_Bundle.zip'
candidates=[Path.cwd()/BUNDLE_NAME,Path('/content')/BUNDLE_NAME,Path('/kaggle/working')/BUNDLE_NAME]
if IN_KAGGLE: candidates += list(Path('/kaggle/input').glob('**/'+BUNDLE_NAME))
bundle=next((p for p in candidates if p.exists()),None)
if bundle is None and IN_COLAB:
    from google.colab import files
    uploaded=files.upload()
    if BUNDLE_NAME not in uploaded: raise FileNotFoundError(BUNDLE_NAME)
    bundle=Path('/content')/BUNDLE_NAME
if bundle is None: raise FileNotFoundError('Upload '+BUNDLE_NAME)
print('bundle SHA256',hashlib.sha256(bundle.read_bytes()).hexdigest())
with zipfile.ZipFile(bundle) as zf: zf.extractall(ROOT)
os.chdir(ROOT)

## Mandatory preflight
Checks source hashes, Python syntax, the P480 certificate, and the exact coefficient identity. It does not start elimination.

In [ ]:
manifest=json.loads(Path('FIN_Singular_Bundle_Manifest.json').read_text())
for name,expected in manifest['sha256'].items():
    actual=hashlib.sha256(Path(name).read_bytes()).hexdigest()
    assert actual==expected,(name,expected,actual)
subprocess.run([sys.executable,'-m','py_compile','fin_phase_exact_algebra.py','fin_program_486.py','fin_singular_campaign.py'],check=True)
import sympy as sp
assert sp.simplify(sp.sqrt(2)-(2-4*sp.sin(sp.pi/8)**2))==0
print('PREFLIGHT PASS')

## Recommended campaign
PREPARE may take several minutes because it performs the exact pivot reduction once. P485 and P487 then run in Singular. Interrupting should produce a stopped checkpoint ZIP.

In [ ]:
command=[sys.executable,'-u','fin_singular_campaign.py','recommended','--root',str(ROOT)]
print(' '.join(command))
completed=subprocess.run(command,cwd=ROOT)
print('exit code',completed.returncode)

## Inspect and download the latest checkpoint

In [ ]:
state=ROOT/'FIN_Singular_Campaign_State.json'
if state.exists(): print(json.dumps(json.loads(state.read_text()),indent=2))
archives=sorted(ROOT.glob('FIN_Singular_Checkpoint_*.zip'),key=lambda p:p.stat().st_mtime)
print([(p.name,p.stat().st_size) for p in archives])
if IN_COLAB and archives:
    from google.colab import files
    files.download(str(archives[-1]))
elif archives: print('Download:',archives[-1])

## Optional P475 — not recommended initially
Run this only after P487 is assessed. It invokes Singular elimination on the original fourteen-variable system and may still exceed the runtime memory/time envelope.

In [ ]:
RUN_P475=False
if RUN_P475:
    subprocess.run([sys.executable,'-u','fin_singular_campaign.py','p475','--root',str(ROOT)],cwd=ROOT)
else:
    print('P475 remains disabled.')